In [1]:
from typing import Any, Dict, List
from dataclasses import dataclass

In [2]:
# define initial test params (same as default web page: https://www.mrcooper.com/calculators/refinance)

original_loan_amount = 100,000
original_term = 30
years_paid = 3
original_interest_rate = 6.5

refi_term = 30
refi_interest_rate = 3
closing_costs_percent = 3.5
finance_closing_costs = "yes"
cash_out_amount = 0


In [3]:
import json
from dataclasses import dataclass
from typing import Dict, Any

@dataclass
class LoanDetails:
    loan_amount: float
    term_years: int
    interest_rate: float

    def monthly_payment(self) -> float:
        monthly_rate = self.interest_rate / 12 / 100
        num_payments = self.term_years * 12
        return (self.loan_amount * monthly_rate * (1 + monthly_rate)**num_payments) / ((1 + monthly_rate)**num_payments - 1)

class LoanRefinanceCalculator:
    """
    A standalone class that calculates loan refinancing options.
    """

    def calculate_refinance(self, current_loan: LoanDetails, refi_loan: LoanDetails, years_paid: int, closing_costs_percent: float, finance_closing_costs: bool, cash_out_amount: float) -> Dict[str, Any]:
        # Ensure all values are float
        loan_amount = float(current_loan.loan_amount)
        term_years = float(current_loan.term_years)
        interest_rate = float(current_loan.interest_rate)
        years_paid = float(years_paid)
        
        # Calculate remaining balance on current loan
        monthly_rate = interest_rate / 12 / 100
        num_payments = term_years * 12
        num_payments_made = years_paid * 12
        
        remaining_balance = loan_amount * ((1 + monthly_rate)**num_payments - (1 + monthly_rate)**num_payments_made) / ((1 + monthly_rate)**num_payments - 1)

        # Calculate new loan amount
        new_loan_amount = remaining_balance + cash_out_amount
        if finance_closing_costs:
            closing_costs = new_loan_amount * closing_costs_percent / 100
            new_loan_amount += closing_costs

        refi_loan.loan_amount = new_loan_amount

        return {
            "original_monthly_payment": round(current_loan.monthly_payment(), 2),
            "refinanced_monthly_payment": round(refi_loan.monthly_payment(), 2),
            "remaining_balance": round(remaining_balance, 2),
            "new_loan_amount": round(new_loan_amount, 2)
        }

    def calculate_from_json(self, loan_data_json: str) -> Dict[str, Any]:
        loan_data = json.loads(loan_data_json)

        original_loan_amount = float(loan_data.get("original_loan_amount", 0))
        original_term = int(loan_data.get("original_term", 0))
        years_paid = int(loan_data.get("years_paid", 0))
        original_interest_rate = float(loan_data.get("original_interest_rate", 0))

        refi_term = int(loan_data.get("refi_term", 0))
        refi_interest_rate = float(loan_data.get("refi_interest_rate", 0))
        closing_costs_percent = float(loan_data.get("closing_costs_percent", 0))
        finance_closing_costs = loan_data.get("finance_closing_costs", "").lower() == "yes"
        cash_out_amount = float(loan_data.get("cash_out_amount", 0))

        current_loan = LoanDetails(original_loan_amount, original_term, original_interest_rate)
        refi_loan = LoanDetails(0, refi_term, refi_interest_rate)  # Loan amount will be calculated in refinance function

        return self.calculate_refinance(current_loan, refi_loan, years_paid, closing_costs_percent, finance_closing_costs, cash_out_amount)

def main():
    calculator = LoanRefinanceCalculator()
    
    # Test case 1: Basic refinance
    loan_data_1 = {
        "original_loan_amount": 100000,
        "original_term": 30,
        "years_paid": 3,
        "original_interest_rate": 6.5,
        "refi_term": 30,
        "refi_interest_rate": 3,
        "closing_costs_percent": 3.5,
        "finance_closing_costs": "yes",
        "cash_out_amount": 0
    }
    

    result_1 = calculator.calculate_from_json(json.dumps(loan_data_1))
    print("Test Case 1 Result:")
    print(json.dumps(result_1, indent=2))
    
    # Test case 2: No cash-out, no financing of closing costs
    loan_data_2 = {
        "original_loan_amount": 300000,
        "original_term": 30,
        "years_paid": 10,
        "original_interest_rate": 5.0,
        "refi_term": 20,
        "refi_interest_rate": 3.5,
        "closing_costs_percent": 1.5,
        "finance_closing_costs": "no",
        "cash_out_amount": 0
    }
    
    result_2 = calculator.calculate_from_json(json.dumps(loan_data_2))
    print("\nTest Case 2 Result:")
    print(json.dumps(result_2, indent=2))

Test Case 1 Result:
{
  "original_monthly_payment": 632.07,
  "refinanced_monthly_payment": 420.73,
  "remaining_balance": 96417.24,
  "new_loan_amount": 99791.85
}

Test Case 2 Result:
{
  "original_monthly_payment": 1610.46,
  "refinanced_monthly_payment": 1415.25,
  "remaining_balance": 244026.19,
  "new_loan_amount": 244026.19
}


In [ ]:
message = """Calculate my loan refinance amount with the following details:
 - original_loan_amount:  100000
- original_term: 30
- years_paid: 3
- original_interest_rate: 6.5
- refi_term: 30
- refi_interest_rate: 3
- closing_costs_percent: 3.5
- finance_closing_costs: yes
- cash_out_amount: 0
"""

In [ ]:
import cohere
import os
co = cohere.Client(api_key=os.environ["COHERE_API_KEY"])

response = co.chat(
   message=message,
   tools=tools,
   # preamble=preamble,
   model="command-r",
   temperature=0.3
)


print("Final answer:")
print(response.text)

In [ ]:
import cohere
co = cohere.Client(api_key=os.environ["COHERE_API_KEY"])

# tool descriptions that the model has access to
tools = [
   {
       "name": "query_daily_sales_report",
       "description": "This tool calculates loan refinancing options based on current loan details and refinancing parameters.",
        "parameter_definitions": {
            "loan_data": {
                "description": "A JSON string containing loan details including original_loan_amount, original_term, years_paid, original_interest_rate, refi_term, refi_interest_rate, closing_costs_percent, finance_closing_costs, and cash_out_amount.",
                "type": "str",
                "required": True,
            }
        },
   },
]

response = co.chat(
   message=message,
   force_single_step=True,
   tools=tools,
#    preamble=preamble,
   model="command-r"
)